# MNIST Fully Connected Network (FCN)

**English:** Homework-style student practice using the public MNIST dataset. Saved Colab execution existed in the original; this copy strips outputs and has not been independently re-run. Model save/load uses legacy .h5 format.

**Tiếng Việt:** Bài thực hành trích từ Colab cá nhân để thể hiện tiến trình học tập. Kết quả chạy trước đây không được sao chép sang bản public; cần kiểm chứng lại phụ thuộc, dữ liệu và từng cell. Không sử dụng mô hình/bài học này để ra quyết định thực tế.

**Provenance / Nguồn:** Coursework/student-practice notebook titled “Bài tập về nhà buổi 7 - Lập trình python.ipynb”. Only code and author-written explanatory cells were retained; no raw execution outputs, account metadata or private data are included.



# Phân loại số viết tay với tập dữ liệu HandWritten MNIST sử dụng Fully Connected Neural Network (FCN)
Tập dữ liệu MNIST bao gồm các ảnh xám 28x28 có nội dung số viết tay từ 0 đến 9. Mục tiêu là huấn luyện một mạng nơ-ron để phân loại chính xác từng ảnh vào một trong các lớp số này.

Gồm các bước sau:
1. Tải tập dữ liệu MNIST.
2. Tiền xử lý dữ liệu.
3. Xây dựng một mạng nơ-ron với lớp FCN.
4. Huấn luyện và đánh giá mô hình.

## loading the MNIST Dataset


In [ ]:
import tensorflow as tf                       # Import thư viện TensorFlow (dùng để xây dựng & huấn luyện mô hình deep learning)
from tensorflow.keras.datasets import mnist   # Import bộ dữ liệu MNIST (chữ số viết tay 0–9, ảnh 28x28)

# Tải dữ liệu MNIST và tự động chia thành 2 phần:
# (x_train, y_train): tập huấn luyện gồm 60,000 ảnh chữ số viết tay và nhãn tương ứng
# (x_test, y_test): tập kiểm tra gồm 10,000 ảnh chữ số viết tay và nhãn tương ứng
(x_train, y_train), (x_test, y_test) = mnist.load_data()

## Pre-processing the Data

In [ ]:
# Normalize the data to 0-1 range
# Khi chia mỗi phần tử cho 255.0, ta biến đổi dữ liệu pixel thành số thực (float) trong khoảng 0 → 1.
x_train = x_train / 255.0 #Ảnh trong MNIST ban đầu được lưu dưới dạng số nguyên (uint8), mỗi pixel có giá trị từ 0 đến 255 (0 = đen, 255 = trắng).
x_test = x_test / 255.0

## Visualizing the Dataset

In [ ]:
import matplotlib.pyplot as plt   # Thư viện vẽ đồ thị và hiển thị hình ảnh
import numpy as np                # Thư viện xử lý số học và mảng (dù ở đây không dùng nhiều)

# In ra kích thước của tập huấn luyện và tập kiểm tra
# x_train: ảnh train, y_train: nhãn train
# x_test: ảnh test, y_test: nhãn test
print(f"Training set shape: {x_train.shape}, Labels shape: {y_train.shape}")
print(f"Test set shape: {x_test.shape}, Labels shape: {y_test.shape}")

# Hàm hiển thị một lưới (grid) ảnh từ dataset
def plot_sample_images(x_data, y_data, num_images=10):
    plt.figure(figsize=(10, 5))  # Tạo khung vẽ với kích thước 10x5 inches
    for i in range(num_images):  # Lặp qua số lượng ảnh muốn hiển thị
        plt.subplot(2, 5, i + 1)  # Chia khung thành 2 hàng 5 cột, chọn ô thứ i+1
        plt.imshow(x_data[i], cmap='gray')  # Hiển thị ảnh i dưới dạng ảnh xám
        plt.title(f"Label: {y_data[i]}")    # Hiển thị nhãn của ảnh lên tiêu đề
        plt.axis('off')  # Ẩn trục X, Y để hình ảnh gọn gàng hơn
    plt.show()  # Xuất hình ảnh ra màn hình

# Hiển thị 10 ảnh đầu tiên trong tập huấn luyện kèm nhãn
plot_sample_images(x_train, y_train)


## Building Neural Network with FCN

In [ ]:
# Xây dựng mô hình mạng nơ-ron nhân tạo (ANN) bằng Keras
from tensorflow.keras import layers, models  # Import các module cần thiết: layers để tạo lớp, models để xây dựng mô hình

# Khởi tạo mô hình Sequential (dạng tuyến tính: lớp này nối tiếp lớp kia)
model = models.Sequential([
    # Lớp Input: xác định đầu vào có kích thước (28, 28, 1)
    # 28x28 là kích thước ảnh, 1 là số kênh màu (ảnh MNIST là grayscale nên kênh = 1)
    layers.InputLayer(input_shape=(28, 28, 1)),

    # Lớp Flatten: biến đổi ảnh 2D (28x28) thành vector 1D (784 phần tử)
    layers.Flatten(),

    # Lớp Dense (fully connected): 128 neuron, hàm kích hoạt ReLU (giúp mô hình học các đặc trưng phi tuyến)
    layers.Dense(128, activation='relu'),

    # Lớp Dense tiếp theo: 64 neuron, hàm kích hoạt ReLU
    layers.Dense(64, activation='relu'),

    # Lớp đầu ra: 10 neuron ứng với 10 lớp (chữ số 0-9), dùng softmax để chuẩn hóa thành xác suất
    layers.Dense(10, activation='softmax')
])

In [ ]:
# Biên dịch (compile) mô hình trước khi huấn luyện
model.compile(
    optimizer='adam',                       # Bộ tối ưu Adam (Adaptive Moment Estimation) giúp quá trình học nhanh và hiệu quả
    loss='sparse_categorical_crossentropy', # Hàm mất mát cho bài toán phân loại nhiều lớp với nhãn là số nguyên (0–9)
    metrics=['accuracy']                    # Chỉ số theo dõi trong quá trình huấn luyện: độ chính xác (accuracy)
)

## Train the Model



In [ ]:
# Huấn luyện (train) mô hình với dữ liệu huấn luyện
model.fit(
    x_train, y_train,                 # Dữ liệu huấn luyện (ảnh và nhãn)
    epochs=10,                        # Số lần lặp (epoch): toàn bộ dữ liệu huấn luyện đi qua mô hình 10 lần
    batch_size=32,                    # Kích thước batch: 32 ảnh được đưa qua mô hình trước khi cập nhật trọng số một lần
    validation_data=(x_test, y_test)  # Bộ dữ liệu kiểm chứng (validation set): dùng để theo dõi loss/accuracy ngoài tập train
)

## Evaluating the Model

In [ ]:
# Đánh giá mô hình trên tập kiểm tra (test set)
test_loss, test_acc = model.evaluate(x_test, y_test)

# In ra độ chính xác (accuracy) trên tập test
print(f"Test accuracy)


## Making Predictions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dùng mô hình đã huấn luyện để dự đoán trên toàn bộ tập test
# Kết quả: mảng 2D (10000 x 10), mỗi hàng là vector xác suất cho 10 lớp
predictions = model.predict(x_test)

# Hàm hiển thị một ảnh test và so sánh nhãn thật với nhãn dự đoán
def plot_image(i, predictions_array, true_label, img):
    plt.grid(False)          # Tắt lưới trên ảnh
    plt.xticks([])           # Tắt trục X
    plt.yticks([])           # Tắt trục Y
    plt.imshow(img[i].reshape(28, 28), cmap=plt.cm.binary)  # Hiển thị ảnh thứ i dưới dạng ảnh đen trắng

    predicted_label = np.argmax(predictions_array[i])  # Lấy nhãn dự đoán (chỉ số có xác suất lớn nhất)
    true_label = true_label[i]                         # Lấy nhãn thật

    # Nếu dự đoán đúng → màu xanh, sai → màu đỏ
    if predicted_label == true_label:
        color = 'blue'
    else:
        color = 'red'

    # Hiển thị nhãn dự đoán và nhãn thật phía dưới ảnh
    # Nếu đúng thì hiện màu xanh, sai thì hiện màu đỏ
    plt.xlabel(f"{predicted_label} ({true_label})", color=color)

# Hiển thị ảnh test đầu tiên (chỉ số 0) kèm nhãn dự đoán
plot_image(0, predictions, y_test, x_test)
plt.show()

## saving the model

In [ ]:
# Lưu mô hình đã huấn luyện
# File được lưu với định dạng HDF5 (.h5), chứa toàn bộ kiến trúc, trọng số và thông tin compile của mô hình
model.save('mnist_fcn_model.h5')

# Khi cần sử dụng lại mô hình (ví dụ để dự đoán, đánh giá hay tiếp tục huấn luyện) → nạp mô hình đã lưu
from tensorflow.keras.models import load_model  # Import hàm load_model để tải mô hình
model = load_model('mnist_fcn_model.h5')        # Nạp lại mô hình từ file đã lưu